In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")
from core.startup import init
engine, memory = init()

In [ ]:
%%writefile /workspace/Projects/cultivated-learning/core/consolidation.py
import time
from core.memory_store import MemoryUnit, MemoryType


class ConsolidationEngine:
    """Distills fading episodic memories into durable semantic memories."""

    def __init__(self, engine, memory_store, salience_threshold=0.4, min_cluster=2):
        self.engine = engine
        self.memory = memory_store
        self.salience_threshold = salience_threshold
        self.min_cluster = min_cluster

    def consolidate(self):
        """Run a consolidation pass. Returns list of new semantic memories created."""
        # Get all episodic memories
        episodic = self.memory.retrieve_by_type(MemoryType.EPISODIC)

        if len(episodic) < self.min_cluster:
            print(f"Consolidation: only {len(episodic)} episodic memories, need {self.min_cluster}. Skipping.")
            return []

        # Split into fading (candidates for consolidation) and fresh
        fading = [m for m in episodic if m.salience_score < self.salience_threshold]
        fresh = [m for m in episodic if m.salience_score >= self.salience_threshold]

        if len(fading) < self.min_cluster:
            print(f"Consolidation: only {len(fading)} fading memories (below {self.salience_threshold}). Skipping.")
            return []

        print(f"Consolidation: {len(fading)} fading episodic memories found. Distilling...")

        # Feed fading memories to the LLM for distillation
        memory_text = "\n\n".join(
            [f"[Episode {i+1}] {m.content}" for i, m in enumerate(fading)]
        )

        prompt = (
            "[INST] You are a memory consolidation module. Your job is to extract "
            "lasting knowledge from a set of interaction episodes.\n\n"
            f"Episodes:\n{memory_text}\n\n"
            "Extract the durable facts, preferences, and patterns from these episodes. "
            "Ignore transient details (greetings, small talk, one-time questions).\n\n"
            "Format: Return one fact per line. Each fact should be a standalone statement "
            "that would be useful to remember long-term. If there are no lasting facts, "
            "respond with exactly: NOTHING_TO_CONSOLIDATE\n\n"
            "Be concise. Maximum 5 facts. [/INST]"
        )

        result = self.engine.generate_structured(prompt, max_new_tokens=300)

        if "NOTHING_TO_CONSOLIDATE" in result.upper():
            print("Consolidation: no lasting knowledge extracted.")
            return []

        # Parse facts into semantic memories
        new_memories = []
        facts = [line.strip() for line in result.strip().split("\n") if line.strip()]

        for fact in facts[:5]:
            # Clean up common LLM formatting
            fact = fact.lstrip("0123456789.-) ").strip()
            if len(fact) < 10:
                continue

            semantic = MemoryUnit(
                content=fact,
                memory_type=MemoryType.SEMANTIC,
                salience_score=0.6,
                confidence=0.7,
                tags=["consolidated", "auto_generated"],
            )
            self.memory.store(semantic)
            new_memories.append(semantic)

        # Mark consolidated episodic memories as superseded
        if new_memories:
            for mem in fading:
                self.memory.adjust_salience(mem.id, -0.2)

        print(f"Consolidation complete: {len(new_memories)} semantic memories created, "
              f"{len(fading)} episodic memories demoted.")
        return new_memories

    def get_consolidation_candidates(self):
        """Preview what would be consolidated without doing it."""
        episodic = self.memory.retrieve_by_type(MemoryType.EPISODIC)
        fading = [m for m in episodic if m.salience_score < self.salience_threshold]
        return fading

In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")

from engine.inference import InferenceEngine
from core.memory_store import MemoryStore
from core.consolidation import ConsolidationEngine

engine = InferenceEngine("/workspace/models/results/Mistral-7B-Instruct-v0.3")
engine.load()

memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

consolidator = ConsolidationEngine(engine, memory)

candidates = consolidator.get_consolidation_candidates()
print(f"\nCandidates for consolidation: {len(candidates)}")
for c in candidates:
    print(f"  [{c.salience_score:.2f}] {c.content[:80]}...")

print("\nRunning consolidation...")
new_mems = consolidator.consolidate()
if new_mems:
    print("\nNew semantic memories:")
    for m in new_mems:
        print(f"  [{m.salience_score:.2f}] {m.content}")

In [ ]:
# Run decay a few times to simulate aging
memory.decay_pass()
memory.decay_pass()
memory.decay_pass()

# Check candidates now
candidates = consolidator.get_consolidation_candidates()
print(f"\nCandidates: {len(candidates)}")
for c in candidates:
    print(f"  [{c.salience_score:.2f}] {c.content[:80]}...")


In [ ]:
from core.memory_store import MemoryType
episodic = memory.retrieve_by_type(MemoryType.EPISODIC)
print(f"\nAll episodic ({len(episodic)}):")
for m in episodic:
    print(f"  [{m.salience_score:.3f}] {m.content[:80]}...")

In [ ]:
memory.decay_pass()

candidates = consolidator.get_consolidation_candidates()
print(f"\nCandidates: {len(candidates)}")
for c in candidates:
    print(f"  [{c.salience_score:.3f}] {c.content[:80]}...")

if candidates:
    print("\nRunning consolidation...")
    new_mems = consolidator.consolidate()
    if new_mems:
        print("\nNew semantic memories:")
        for m in new_mems:
            print(f"  [{m.salience_score:.2f}] {m.content}")

In [ ]:
episodic = memory.retrieve_by_type(MemoryType.EPISODIC)
lowest = sorted(episodic, key=lambda m: m.salience_score)[:5]
for m in lowest:
    print(f"  [{m.salience_score:.4f}] {m.content[:80]}...")

In [ ]:
consolidator.salience_threshold = 0.41

candidates = consolidator.get_consolidation_candidates()
print(f"Candidates: {len(candidates)}")
for c in candidates:
    print(f"  [{c.salience_score:.4f}] {c.content[:80]}...")

if candidates:
    print("\nRunning consolidation...")
    new_mems = consolidator.consolidate()
    if new_mems:
        print("\nNew semantic memories:")
        for m in new_mems:
            print(f"  [{m.salience_score:.2f}] {m.content}")

# Reset threshold
consolidator.salience_threshold = 0.4

In [ ]:
consolidator.salience_threshold = 0.51
consolidator.min_cluster = 2

candidates = consolidator.get_consolidation_candidates()
print(f"Candidates: {len(candidates)}")
for c in candidates:
    print(f"  [{c.salience_score:.4f}] {c.content[:60]}...")

if len(candidates) >= 2:
    print("\nRunning consolidation...")
    new_mems = consolidator.consolidate()
    if new_mems:
        print("\nNew semantic memories:")
        for m in new_mems:
            print(f"  [{m.salience_score:.2f}] {m.content}")

# Reset
consolidator.salience_threshold = 0.4
consolidator.min_cluster = 2

In [ ]:
import json

with open("/workspace/Projects/cultivated-learning/data/memory_backup.json", "r") as f:
    backup = json.load(f)

print("Original salience values:")
for item in backup:
    mem_id = item["id"]
    original_sal = item["metadata"]["salience_score"]
    # Get current salience
    current = memory.collection.get(ids=[mem_id])
    if current["ids"]:
        current_sal = current["metadatas"][0]["salience_score"]
        delta = original_sal - current_sal
        if abs(delta) > 0.001:
            memory.adjust_salience(mem_id, delta)
            print(f"  Reset: {item['document'][:50]}... {current_sal:.3f} → {original_sal:.3f}")
        else:
            print(f"  OK: {item['document'][:50]}... {original_sal:.3f}")
    else:
        print(f"  Missing: {item['document'][:50]}...")

print("\nOriginal 8 memories restored.")